# Experiment with various Transformer based models

In [ ]:
# Standard library
from __future__ import annotations

import copy
import gc
import os
import random
from datetime import datetime
import time
from typing import Any, Protocol
from pathlib import Path

# Third-party libraries
from collections import Counter
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
import numpy as np
import optuna
import pandas as pd
import polars as pl
import seaborn as sns
import shap
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn


# Hugging Face / NLP
from datasets import Dataset, DatasetDict, concatenate_datasets

from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BatchEncoding,
    EarlyStoppingCallback,
    EvalPrediction,
    Trainer,
    TrainingArguments,
    pipeline,
)

# Scikit-learn
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

# Project-specific imports
from ixbrl_ai.data import get_split
from ixbrl_ai.display import display_wide, heading
from ixbrl_ai.sample import DataSample

from ixbrl_ai.test import (
    coerce_to_label_array,
    test_model_over_populations,
    test_model_over_populations_nn,
    log_population_test_results_to_mlflow,
    load_population_test_results_from_mlflow,
    IXBRL_TEXT_CLASSIFICATION_TEST_CASES
)

%load_ext autoreload
%autoreload 2

# 1. Config, setup mlflow, load data

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Set random seed
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

device_type = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else device_type)
print(f"Device selected is {DEVICE}")

experiment_name = "sentence-transformers-compare"
mlflow.set_experiment(experiment_name)
mlflow.transformers.autolog()

# mpnet might be better for longer text, mini might be faster and do just as well
MODELS = [
    "roberta-base",
    "nlpaueb/sec-bert-base",
    "sentence-transformers/all-mpnet-base-v2",
    "sentence-transformers/all-MiniLM-L6-v2",
]

In [ ]:
X = "canonical_description"
y = "label"

def tokenize(batch: dict[str, list[Any]], model_name: str,max_length: int = 32) -> BatchEncoding:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    return tokenizer(batch[X], padding="max_length", truncation=True, max_length=max_length)


def encode(data_pl: pl.DataFrame, model_name: str, max_length: int = 32) -> Dataset:
    dataset = Dataset.from_polars(data_pl.select(X, y))
    dataset_encoded = dataset.map(lambda batch: tokenize(batch, model_name=model_name, max_length=max_length), batched=True)
    dataset_encoded = dataset_encoded.remove_columns([X])
    dataset_encoded.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    return dataset_encoded



def tokenize_for_model(model_name: str, datasets: dict[str, pl.DataFrame], max_length: int = 64) -> dict[str, Dataset]:
    """Tokenizes dataset using model

    Args:
        model_name (str): Model name
        datasets (dict[str, pl.DataFrame]): The datasets to tokenize
        max_length (int, optional): Max token length. Defaults to 64.

    Returns:
        dict[str, Dataset]: Tokenized datasets
    """
    

    return {key: encode(value, model_name=model_name, max_length=max_length) for key, value in datasets.items()}


def cache_slug(value: str) -> str:
    """Converts a string to a slug suitable for caching by replacing certain characters.
    
    Args:
        value (str): The input string to be converted into a slug.

    Returns:
        str: The converted slug string.
    """
    return value.replace("/", "__").replace(" ", "_")

def save_encoded_datasets(datasets_encoded: dict[str, dict[str, Dataset]], dataset_name: str) -> None:
    """Saves encoded datasets to disk for the specified dataset name.
    Args:
        datasets_encoded (dict[str, dict[str, Dataset]]): A dictionary mapping each model name to its corresponding dictionary of split names and encoded Datasets.
        dataset_name (str): The name of the dataset version to save.    
    """
    
    datasets_encoded_dict = {model: DatasetDict({split: dataset for split, dataset in encoded_datasets.items()}) for model, encoded_datasets in datasets_encoded.items()}   
    for model, encoded in datasets_encoded_dict.items():
        encoded.save_to_disk(f"data/tokenized_transformers/{dataset_name}/{cache_slug(model)}")

def load_encoded_datasets(models: list[str], dataset_name: str) -> dict[str, DatasetDict]:
    """Loads encoded datasets from disk for the specified models and dataset name.

    Args:
        models (list[str]): A list of model names for which to load the encoded datasets.
        dataset_name (str): The name of the dataset version to load.    
    Returns:
        dict[str, DatasetDict]: A dictionary mapping each model name to its corresponding loaded DatasetDict.
    """
    return {
        model: DatasetDict.load_from_disk(f"data/tokenized_transformers/{dataset_name}/{cache_slug(model)}") 
        for model in models
        }

In [ ]:
dataset_version = "v13"
dataset_name = f"data/canonicalized_split_{dataset_version}.parquet"
dataset_pl = pl.read_parquet(dataset_name)
subset = DataSample.sample_1_pct

datasets = {
    "train": dataset_pl.filter(pl.col("train")),
    DataSample.sample_1_pct.label: dataset_pl.filter(pl.col(DataSample.sample_1_pct.label)),
    DataSample.sample_10_pct.label: dataset_pl.filter(pl.col(DataSample.sample_10_pct.label)),
    DataSample.sample_100_pct.label: dataset_pl.filter(pl.col(DataSample.sample_100_pct.label)),
    "sample_1_pct_sqrt_weight": dataset_pl.filter(pl.col("sample_1_pct_sqrt_weight")),
    "sample_10_pct_sqrt_weight": dataset_pl.filter(pl.col("sample_10_pct_sqrt_weight")),
    "test": dataset_pl.filter(pl.col("test")),
    "test_5_pct": dataset_pl.filter(pl.col("test_5_pct")),
    "holdout": dataset_pl.filter(pl.col("holdout")),
}

# Some model names are linked to dataset version, use a seperate testing dataset with smaller holdout so it doesn't crash.
dataset_test_pl = pl.read_parquet("data/canonicalized_split_v16.parquet")
subset = DataSample.sample_1_pct

datasets_test = {
    "train": dataset_test_pl.filter(pl.col("train")),
    DataSample.sample_1_pct.label: dataset_test_pl.filter(pl.col(DataSample.sample_1_pct.label)),
    DataSample.sample_10_pct.label: dataset_test_pl.filter(pl.col(DataSample.sample_10_pct.label)),
    DataSample.sample_100_pct.label: dataset_test_pl.filter(pl.col(DataSample.sample_100_pct.label)),
    "sample_1_pct_sqrt_weight": dataset_test_pl.filter(pl.col("sample_1_pct_sqrt_weight")),
    "sample_10_pct_sqrt_weight": dataset_test_pl.filter(pl.col("sample_10_pct_sqrt_weight")),
    "sample_50_pct_sqrt_weight": dataset_test_pl.filter(pl.col("sample_50_pct_sqrt_weight")),
    "test": dataset_test_pl.filter(pl.col("test")),
    "test_5_pct": dataset_test_pl.filter(pl.col("test_5_pct")),
    "holdout": dataset_test_pl.filter(pl.col("holdout")),
    "holdout_10k": dataset_test_pl.filter(pl.col("holdout_10k")),
    "holdout_10_pct": dataset_test_pl.filter(pl.col("holdout_10_pct")),
}

## 1.1 Check token length

In [ ]:
datasets_encoded = {model: tokenize_for_model(model, datasets) for model in MODELS}

In [ ]:
# Characters
dataset_pl.select(pl.col(X).str.len_chars().max()).pipe(display)
# Words
dataset_pl.select(pl.col(X).str.split(" ").list.len().max())

In [ ]:
all_lengths = []
for model_name in ["nlpaueb/sec-bert-base", "roberta-base", "sentence-transformers/all-mpnet-base-v2"]:
    for split in ["train", "test", "holdout"]:
        all_lengths += [sum(mask) for mask in datasets_encoded[model_name][split]["attention_mask"]]

print(max(all_lengths))

max_length was 32, so setting it to 32 should be fine. If there is a very rare occasion where there are more than 32 tokens, it's fine to truncate it. 

## 1.2 Encode data

In [ ]:
# datasets_encoded = {model: tokenize_for_model(model_name=model, datasets=datasets, max_length=32) for model in MODELS}
# save_encoded_datasets(datasets_encoded, dataset_version)
datasets_encoded = load_encoded_datasets(MODELS, dataset_version)

# datasets_test_encoded = {model: tokenize_for_model(model_name=model, datasets=datasets_test, max_length=32) for model in MODELS}
# save_encoded_datasets(datasets_test_encoded, "v16")
datasets_test_encoded = load_encoded_datasets(MODELS, "v16")

In [ ]:
# Do this over the full dataset, since you need continuous label's, using train messes it up completely
# The models require these kinds of dicts rather than LabelEncoder, so just use them for everything
unique_pl = dataset_pl.select("canonical_label", "label").unique().sort("label")
id2label = dict(zip(unique_pl["label"], unique_pl["canonical_label"]))
label2id = dict(zip(unique_pl["canonical_label"], unique_pl["label"]))
num_labels=unique_pl.height

## 1.3 Functions

In [ ]:
def compute_metrics(eval_pred: EvalPrediction) -> dict[str, Any]:
    """Computes macro metrics

    Args:
        eval_pred (EvalPrediction): Predictions

    Returns:
        dict[str, float]: Metrics
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0,
    )
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1_macro": f1,
    }


def load_best_trainer(experiment_name: str, run_name: str) -> tuple[Trainer, str, dict]:
    """Loads the best trainer from run and returns original model name and args

    Args:
        run_name (str): Run rame

    Returns:
        tuple[Trainer, str]: Return best trainer, model name and args
    """
    client = mlflow.tracking.MlflowClient()
    exp = client.get_experiment_by_name(experiment_name)
    parent_runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'",
    )
    parent_run = parent_runs[0]

    child_runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.parentRunId = '{parent_run.info.run_id}'",
        order_by=["metrics.eval_f1_macro DESC"],
    )
    best_run = child_runs[0]
    model_path = f"/Volumes/WDElement/ML/EPA/bert/{run_name}/{best_run.info.run_name}"

    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    training_args = torch.load(f"{model_path}/training_args.bin", weights_only=False)
    original_args = copy.deepcopy(training_args)
    training_args.eval_strategy = "no"
    initial_model = best_run.data.params["_name_or_path"]
    print(f"model_path: {model_path}")
    print(f"model_type: {initial_model}")
    print(f"model_args {training_args}")
    return Trainer(model=model, args=training_args), initial_model, original_args


def get_run_details(experiment_name: str, run_name: str, index: int = 0) -> pl.DataFrame:
    """Puts run details into a dataframe

    Args:
        experiment_name (str): Experiment name
        run_name (str): Run rame

    Returns:
        pl.DataFrame: Dataframe with metrics and params
    """

    client = mlflow.tracking.MlflowClient()
    exp = client.get_experiment_by_name(experiment_name)
    parent_runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.runName = '{run_name}'",
    )
    parent_run = parent_runs[index]

    child_runs = client.search_runs(
        experiment_ids=[exp.experiment_id],
        filter_string=f"tags.mlflow.parentRunId = '{parent_run.info.run_id}'",
        order_by=["metrics.eval_f1_macro DESC"],
    )

    runs_data = []
    for run in child_runs:
        run_data = {
            "model": run.data.params["_name_or_path"],
            "run": run.info.run_name,
            **run.data.metrics,
            **run.data.params,
        }
        runs_data.append(run_data)

    return pl.DataFrame(runs_data).with_row_index("rank")


def test_predictions_batch(
    trainer: Trainer,
    model_name: str,
    datasets_encoded: dict[str, dict[str, Dataset]],
    test_name: str,
    batch_size: int = 20000,
) -> dict[str, float]:
    """Prints prediction metrics using batch prediction

    Args:
        trainer (Trainer): HuggingFace Trainer
        model_name (str): Model name
        datasets_encoded (dict[str, dict[str, Dataset]]): Nested dict with the datasets
        test_name (str): Key used for dataset to test against
        batch_size (int): Batch size for prediction
    """
    dataset_test = datasets_encoded[model_name][test_name]
    predictions = []
    labels = []
    for i in range(0, len(dataset_test), batch_size):
        batch = dataset_test.select(range(i, min(i + batch_size, len(dataset_test))))
        prediction_options = trainer.predict(batch)
        predictions.extend(np.argmax(prediction_options.predictions, axis=1))
        labels.extend(prediction_options.label_ids)
        del prediction_options
        gc.collect()
        torch.mps.empty_cache()

    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="macro")
    metrics = {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1_macro": f1,
    }

    for key, value in metrics.items():
        print(f"{key}: {value}")

    return metrics


def test_predictions(
    trainer: Trainer,
    model_name: str,
    datasets_encoded: dict[str, dict[str, Dataset]],
    test_name: str,
) -> dict[str, float]:
    """Prints prediction metrics

    Args:
        trainer (Trainer): HuggingFace Trainer
        model_name (str): Model name
        datasets_encoded (dict[str, dict[str, Dataset]]): Nested dict with the datasets
        test_name (str): Key used for dataset to test against
    """
    test_encoded = datasets_encoded[model_name][test_name]
    prediction_options = trainer.predict(test_encoded)
    metrics = compute_metrics(
        EvalPrediction(
            predictions=prediction_options.predictions,
            label_ids=prediction_options.label_ids,
        )
    )

    for key, value in metrics.items():
        print(f"{key}: {value}")

    del prediction_options
    gc.collect()
    torch.mps.empty_cache()

    return metrics

def get_run_name(run_prefix: str, train_subset: str) -> str:
    """Generates a run name based on the run prefix and training subset.

    Args:
        run_prefix (str): The prefix for the run name, typically indicating the experiment or model type.
        train_subset (str): The specific training subset used, which will be included in the run name.

    Returns:
        str: A formatted run name combining the prefix and training subset.
    """
    return f"{run_prefix}_{train_subset}"

def train_model(
    model_name: str,
    training_args: dict,
    train_subset: str,
    run_prefix: str,
    num_train_epochs: int | None = None,
) -> Trainer:
    """Trains a model using HuggingFace Trainer and logs the run to MLflow.

    Args:
        model_name (str): The name of the model to be trained, which should correspond to the keys in the datasets_encoded dictionary.
        training_args (dict): A dictionary of training arguments to be passed to the Trainer, such as learning rate, batch size, etc.
        train_subset (str): The specific training subset to be used for training, which should correspond to the keys in the datasets_encoded dictionary.
        run_prefix (str): A prefix for the MLflow run name, typically indicating the experiment or model type.
        num_train_epochs (int | None, optional): The number of training epochs to override in the training arguments. If None, the number of epochs specified in training_args will be used. Defaults to None.

    Returns:
        Trainer: The trained HuggingFace Trainer instance.
    """

    run_name = get_run_name(run_prefix, train_subset)
    output_dir = f"/Volumes/WDElement/ML/EPA/bert/{run_name}/trial_0"

    with mlflow.start_run(run_name=run_name) as parent_run:
        with mlflow.start_run(run_name="trial_0", nested=True):
            training_args.output_dir = output_dir

            if num_train_epochs is not None:
                training_args.num_train_epochs = num_train_epochs

            model = AutoModelForSequenceClassification.from_pretrained(
                model_name,
                num_labels=num_labels,
                id2label=id2label,
                label2id=label2id,
            )

            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=datasets_encoded[model_name][train_subset],
                eval_dataset=datasets_encoded[model_name]["test_5_pct"],
                compute_metrics=compute_metrics,
                callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
            )

            mlflow.log_params(training_args.to_dict())
            mlflow.log_param("train_subset", train_subset)
            start_time = datetime.now()
            trainer.train()
            end_time = datetime.now()
            best_metric = trainer.state.best_metric
            mlflow.log_metric("best_metric", best_metric)
            duration = (end_time - start_time).total_seconds()
            print(f"duration: {duration}s")
            mlflow.log_metric("train_time", duration)
            trainer.save_model(output_dir)
            AutoTokenizer.from_pretrained(model_name).save_pretrained(output_dir)

    return trainer


def load_trainer(model_name: str, run_prefix: str, train_subset: str) -> Trainer:
    """Loads best model from run and tests it

    Args:
        model_name (str): Model name
        run_prefix (str): Run prefix
        train_subset (str): Training subset used

    Returns:
        Trainer: Loaded trainer
    """
    model_dir = f"/Volumes/WDElement/ML/EPA/bert/{run_prefix}_{train_subset}/trial_0"

    trainer = Trainer(
        model=AutoModelForSequenceClassification.from_pretrained(model_dir),
        args=TrainingArguments(per_device_eval_batch_size=128, eval_accumulation_steps=8),
    )
    return trainer

# 2. Optuna compare different models and hyperparameters

In [ ]:
def objective(trial: optuna.Trial) -> float:
    """Optuna objective function for hyperparameter optimization.
    Args:        
        trial (optuna.Trial): The Optuna trial object that provides methods for suggesting hyperparameters and logging results.
    Returns:
        float: The best metric (e.g., F1 score) achieved during training, which Optuna will use to evaluate the trial's performance.
    """

    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        model_name = trial.suggest_categorical("model", MODELS)
        # dropout=trial.suggest_float("dropout", 0.1, 0.4)

        config = AutoConfig.from_pretrained(
            model_name,
            num_labels=num_labels,
            id2label=id2label,
            label2id=label2id,
        )
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            config=config,
        )
        model = model.to(DEVICE)

        output_dir = f"/Volumes/WDElement/ML/EPA/bert/{run_name}/trial_{trial.number}"
        training_args = TrainingArguments(
            output_dir=output_dir,
            per_device_train_batch_size=trial.suggest_categorical("per_device_train_batch_size", [16, 32, 64]),
            per_device_eval_batch_size=64,
            warmup_ratio=trial.suggest_float("warmup_ratio", 0.0, 0.3),
            learning_rate=trial.suggest_float("learning_rate", 1e-6, 1e-4, log=True),
            weight_decay=trial.suggest_float("weight_decay", 0.0, 0.1),
            lr_scheduler_type=trial.suggest_categorical("lr_scheduler_type", ["linear", "cosine"]),
            num_train_epochs=6,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            logging_steps=100,
            logging_dir="logs",
            load_best_model_at_end=True,
            metric_for_best_model="eval_f1_macro",
            greater_is_better=True,
            fp16=False,
            bf16=False,
            dataloader_pin_memory=False,
        )

        # trainer = WeightedTrainer(
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=datasets_encoded[model_name][subset.label],
            eval_dataset=datasets_encoded[model_name]["test_5_pct"],
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        )

        mlflow.log_params(trial.params)
        start_time = datetime.now()
        trainer.train()
        end_time = datetime.now()
        best_metric = trainer.state.best_metric
        mlflow.log_metric("best_metric", best_metric)
        # mlflow.log_param("model_name", model_name)
        # mlflow.log_param("trial_number", trial.number)
        # mlflow.log_param("dropout", dropout)
        mlflow.log_metric("train_time", (end_time - start_time).total_seconds())
        trainer.save_model(output_dir)
        AutoTokenizer.from_pretrained(model_name).save_pretrained(output_dir)

        del model
        del trainer

        gc.collect()
        torch.mps.empty_cache()

    return best_metric


class SupportsPredict(Protocol):
    def predict(self, X: Any) -> Any: ...

def predict_fn(texts: list[str], model: SupportsPredict, id2label: dict) -> pl.DataFrame:
    """Predict labels for a list of texts
    
    Args:
        texts (list[str]): List of texts to classify.
        id2label (dict): Mapping from label IDs to label names.

    Returns:
        pl.DataFrame: A Polars DataFrame with the input texts and their predicted labels.
    """
    texts_encoded = encode(pl.DataFrame({"canonical_description": texts, "label": [0] * len(texts)}), model_name=model_name, max_length=32)
    predictions = model.predict(texts_encoded)
    predicted_labels = [id2label[id] for id in coerce_to_label_array(predictions.predictions)]
    return predicted_labels

In [ ]:
run_name="bert_models_compared_no_weighting_fixed"
subset = DataSample.sample_1_pct

In [ ]:
with mlflow.start_run(run_name=run_name) as parent_run:
    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
        # Warmup steps is warmup epochs
        pruner=optuna.pruners.MedianPruner(n_startup_trials=12, n_warmup_steps=2),
        study_name=run_name,
    )
    global_start_time = datetime.now()
    study.optimize(objective, n_trials=80)
    global_end_time = datetime.now()
    print(f"Total duration {(global_end_time - global_start_time).total_seconds()}s")
    print(f"Best trail: {study.best_trial.params}")

## 2.1 Load best trainer and args

In [ ]:
trainer, model_name, training_args = load_best_trainer(experiment_name=experiment_name, run_name=run_name)

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_test_encoded, test_name="holdout_10k")

In [ ]:
# Don't use they uses too much memory.
# test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test")
# test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="holdout")

- This uses 150GB, if doing again or much more need to write it so it does it in batches and just save the argmax.
- The actual test results are very close to the test_5_pct population, so just use that.

In [ ]:
history_pl = get_run_details(experiment_name=experiment_name, run_name=run_name, index=1)

In [ ]:
display_wide(history_pl, rows=1000)

In [ ]:
run_pl = get_run_details(experiment_name=experiment_name, run_name=run_name)

In [ ]:
run_pl.filter(pl.col("rank") == pl.col("rank").min().over("model"))

- sec-bert-base had the best f1_macro with 0.754, with train time 567s
- roberta-base second 0.743, with train time 460s - slight less performance but a bit faster and more "reputable" model
- mpnet 0.714, with train time 569s
- mini 0.681, with train time 170s - Best speed but worse performance.

# 3. Candidate model

## 3.1 1% train population

### 3.1.1 Default

In [ ]:
trainer, model_name, training_args = load_best_trainer(experiment_name=experiment_name, run_name='bert_models_compared_no_weighting_fixed')

In [ ]:
trainer = train_model(
    model_name=model_name, training_args=training_args, train_subset="train", run_prefix="Final_model"
)

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

### 3.1.2 Batch 64

In [ ]:
training_args.per_device_train_batch_size = 64
trainer = train_model(
    model_name=model_name,
    training_args=training_args,
    train_subset=DataSample.sample_1_pct.label,
    run_prefix="Final_model",
)

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

### 3.1.3 Batch 32

In [ ]:
training_args.per_device_train_batch_size = 32
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_1_pct.label, 
                      run_prefix="Final_model",
                      num_train_epochs=10,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

### 3.1.4 Batch 16

In [ ]:
training_args.per_device_train_batch_size = 16
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_1_pct.label, 
                      run_prefix="Final_model",
                      num_train_epochs=10,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

## 3.2 10% train population

### 3.2.1 Default

### 3.2.1 Batch 32

In [ ]:
training_args.per_device_train_batch_size = 32
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_10_pct.label, 
                      run_prefix="Final_model",
                      num_train_epochs=10,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

### 3.2.1 Batch 16

In [ ]:
training_args.per_device_train_batch_size = 16
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_10_pct.label, 
                      run_prefix="Final_model_batch_16",
                      num_train_epochs=10,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_test_encoded, test_name="holdout_10k")

In [ ]:
trainer = load_trainer(model_name=model_name, run_prefix="Final_model_batch_16", train_subset=DataSample.sample_10_pct.label)

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

On 10% population batch size 16 performed better on f1 macro than batch size 32

## 3.3 100% train population

### 3.3.1 Batch 32

In [ ]:
training_args.per_device_train_batch_size = 32
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_100_pct.label, 
                      run_prefix="Final_model",
                      num_train_epochs=10,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

In [ ]:
trainer = load_trainer(model_name=model_name, run_prefix="Final_model", train_subset=DataSample.sample_100_pct.label)

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_test_encoded, test_name="holdout_10k")

### 3.3.2 Batch 16

In [ ]:
training_args.per_device_train_batch_size = 16
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_100_pct.label, 
                      run_prefix="Final_model_batch_16",
                      num_train_epochs=10,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_test_encoded, test_name="holdout_10k")

Batch 16 has best f1-macro score of 0.782 vs 0.767 but took 14.3h vs 4.4h to train

## 3.4 1% sqrt weight train population

### 3.4.1 Batch 16

In [ ]:
training_args.per_device_train_batch_size = 16
trainer = train_model(
    model_name=model_name,
    training_args=training_args,
    train_subset="sample_1_pct_sqrt_weight",
    run_prefix="Final_model",
    num_train_epochs=10,
)

In [ ]:
trainer = load_trainer(model_name=model_name, run_prefix="Final_model", train_subset="sample_1_pct_sqrt_weight")

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

- VS plain 1% population 
    - F1-macro of 0.7792 is better than 0.7510
- VS plain 10% population
    - F1-macro of 0.7792 is better than 0.7663
    - Accuracy 0.9746 is worse than 0.9757
    - Train time 11.2min is 4 times quicker than 44.4min
- VS 100% population
    - Worse accuracy and F1-macro

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_test_encoded, test_name="holdout_10k")

## 3.5 10% sqrt weight training population

### 3.5.1 Default

In [ ]:
trainer = train_model(model_name=model_name, training_args=training_args, train_subset=DataSample.sample_10_pct.label, run_prefix="Final_model")

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

### 3.5.1 Batch 16

In [ ]:
training_args.per_device_train_batch_size = 16
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_10_pct_sqrt_weight.label, 
                      run_prefix="Final_model_15_epochs",
                      num_train_epochs=15,
                      )



In [ ]:
trainer, model_name, training_args = load_best_trainer(experiment_name=experiment_name, run_name='bert_models_compared_no_weighting_fixed')

In [ ]:
run_prefix = "Final_model_15_epochs"
train_subset = DataSample.sample_10_pct_sqrt_weight.label
trainer = load_trainer(model_name=model_name, run_prefix=run_prefix, train_subset=train_subset)
run_name = get_run_name(run_prefix, train_subset)

In [ ]:
model_dir = f"/Volumes/WDElement/ML/EPA/bert/{run_prefix}_{train_subset}/trial_0"


In [ ]:
model_size= sum(
    f.stat().st_size
    for f in Path(model_dir).rglob("*")
    if f.is_file()
)

In [ ]:
datasets_test_encoded[model_name]["test"]

In [ ]:
data_X = datasets_test_encoded[model_name]["test"]
start = time.perf_counter()
preds = trainer.predict(data_X)
end = time.perf_counter()
inference_time = end - start

In [ ]:
results = test_model_over_populations_nn(
    model=trainer,
    dataset=datasets_test_encoded[model_name])

In [ ]:

evaluation_run_id = log_population_test_results_to_mlflow(
    results,
    experiment_name=experiment_name,
    source_run_name=run_name,
    run_name=f"{run_name}_population_eval",
    dataset_name=dataset_name,
    subset=subset.label,
    train_time=4023,
    model_size=model_size,
    inference_time=inference_time,
)

evaluation_run, saved_results = load_population_test_results_from_mlflow(
    experiment_name=experiment_name,
    source_run_name=run_name,
)

display(saved_results)

In [ ]:
test_cases_pl = pl.DataFrame(IXBRL_TEXT_CLASSIFICATION_TEST_CASES)
predictions = predict_fn(test_cases_pl["text"].to_list(), model=trainer, id2label=id2label)
test_cases_pl = (
    test_cases_pl
    .with_columns(pl.Series("predicted_label", predictions))
    .with_columns(((pl.col("predicted_label") == pl.col("expected")) == pl.col("should_match")).alias("correct"))
)

In [ ]:
test_cases_pl.write_parquet("data/nn_ixbrl_text_classification_test_cases_with_predictions.parquet")

In [ ]:
cat_pl = test_cases_pl.group_by("category").agg(
    total_cases=pl.len(),
    correct_predictions=(pl.col("correct")).sum()
).with_columns(
    accuracy=(pl.col("correct_predictions") / pl.col("total_cases"))
).sort("category")
cat_pl.pipe(display_wide)

display(cat_pl["accuracy"].sum())

exp_pl = test_cases_pl.group_by("expected").agg(
    total_cases=pl.len(),
    correct_predictions=(pl.col("correct")).sum()
).with_columns(
    accuracy=(pl.col("correct_predictions") / pl.col("total_cases"))    
).sort("expected")
exp_pl.pipe(display_wide)

display(exp_pl["accuracy"].sum())

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="holdout_10k")

- VS plain 10% population using 5% test data set
    - F1-macro 0.7785 better than 0.7663
    - Accuracy 0.9743 worse than 0.9757

On the test_5_pct dataset 100% population did better on both f1-macro and accuracy.   
But on holdout_10k population f1-macro was better than 100% population with a very small difference on accuracy. 

But considering that it's on a much smaller population it's about 14 times faster, weighting of training population is effective, but it does add complexity, and you can't realy do that over the full population. Ideally production models would be trained over as much data as possible.

In [ ]:
results = bootstrap_test_predictions(
    trainer=trainer,
    model_name=model_name,
    datasets_encoded=datasets_test_encoded,
    test_name="holdout_10k",
)

## 3.6 50% sqrt weight training population

In [ ]:
datasets_encoded = datasets_test_encoded

In [ ]:
training_args.per_device_train_batch_size = 16
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_50_pct_sqrt_weight.label, 
                      run_prefix="Final_model_15_epochs",
                      num_train_epochs=15,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="holdout_10k")

VS 100% population
- 8.3h is quicker than 14.3h
- Accuracy 0.9766 is a touch better than 0.9765
- F1-macro 0.7844 is better than 0.7819

So you get better metrics and it's faster to use sqrt weighted data.

## 3.7 Test Weighted model

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None
        )

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss


def get_class_weights(train_dataset, num_labels: int) -> torch.Tensor:
    labels = np.array(train_dataset["label"])
    present_classes = np.unique(labels)

    present_weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=labels,
    )

    # full weight vector for all labels
    full_weights = np.ones(num_labels, dtype=np.float32)

    # assign computed weights only to classes present in this subset
    for cls, weight in zip(present_classes, present_weights):
        full_weights[int(cls)] = float(weight)

    return torch.tensor(full_weights, dtype=torch.float)


def get_class_weights_sqrt(train_dataset, num_labels: int) -> torch.Tensor:
    labels = np.array(train_dataset["label"])
    present_classes = np.unique(labels)

    present_weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=labels,
    )

    # full weight vector for all labels
    full_weights = np.ones(num_labels, dtype=np.float32)

    # assign computed weights only to classes present in this subset
    for cls, weight in zip(present_classes, present_weights):
        full_weights[int(cls)] = float(np.sqrt(weight))

    return torch.tensor(full_weights, dtype=torch.float)


def train_model(
    model_name: str,
    training_args,
    train_subset: str,
    run_prefix: str,
    train_dataset: Dataset | None = None,
    num_train_epochs: int | None = None,
    sqrt_weight: bool = False,
) -> Trainer:

    run_name = f"{run_prefix}_{train_subset}"
    output_dir = f"/Volumes/WDElement/ML/EPA/bert/{run_name}/trial_0"

    if train_dataset is None:
        train_dataset = datasets_encoded[model_name][train_subset]
    eval_dataset = datasets_encoded[model_name]["test_5_pct"]

    class_weights = get_class_weights_sqrt(train_dataset, num_labels=num_labels) if sqrt_weight else get_class_weights(train_dataset, num_labels=num_labels)

    with mlflow.start_run(run_name=run_name) as parent_run:
        with mlflow.start_run(run_name="trial_0", nested=True):
            training_args.output_dir = output_dir

            if num_train_epochs is not None:
                training_args.num_train_epochs = num_train_epochs

            # Make sure best checkpoint selection uses macro F1
            training_args.metric_for_best_model = "f1_macro"
            training_args.load_best_model_at_end = True
            training_args.greater_is_better = True

            model = AutoModelForSequenceClassification.from_pretrained(
                model_name,
                num_labels=num_labels,
                id2label=id2label,
                label2id=label2id,
            )

            trainer = WeightedTrainer(
                model=model,
                args=training_args,
                train_dataset=train_dataset,
                eval_dataset=eval_dataset,
                compute_metrics=compute_metrics,
                callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
                class_weights=class_weights,
            )

            mlflow.log_params(training_args.to_dict())
            mlflow.log_param("train_subset", train_subset)

            for i, w in enumerate(class_weights.tolist()):
                mlflow.log_metric(f"class_weight_{i}", w)

            start_time = datetime.now()
            trainer.train()
            end_time = datetime.now()

            duration = (end_time - start_time).total_seconds()
            print(f"duration: {duration}s")

            mlflow.log_metric("train_time", duration)

            if trainer.state.best_metric is not None:
                mlflow.log_metric("best_metric", trainer.state.best_metric)

            trainer.save_model(output_dir)
            AutoTokenizer.from_pretrained(model_name).save_pretrained(output_dir)

    return trainer

### 3.7.1 1% train population

In [ ]:
training_args.per_device_train_batch_size = 16
training_args.per_device_eval_batch_size = 16
training_args.num_train_epochs = 10
training_args.metric_for_best_model = "f1_macro"
training_args.load_best_model_at_end = True
training_args.greater_is_better = True
training_args.evaluation_strategy = "epoch"
training_args.save_strategy = "epoch"
training_args.logging_strategy = "epoch"
training_args.save_total_limit = 2

In [ ]:
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_1_pct.label, 
                      run_prefix="Final_model",
                      num_train_epochs=10,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

F1-macro score was worse

### 3.7.2 10% train population

In [ ]:
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_10_pct.label, 
                      run_prefix="Final_model",
                      num_train_epochs=15,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="holdout_10k")

### 3.7.3 10% sqrt weighted train population

In [ ]:
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_10_pct_sqrt_weight.label, 
                      run_prefix="Final_model_weighted",
                      num_train_epochs=15,
                      )

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_test_encoded, test_name="holdout_10k")

Interestingly weighted model and weighted data has worse F1-macro score than just the weighted model of 10% population.  
Overall performance accuracy and f1-macro were worse than the normal models.

## 3.8 Sqrt Weighted model

In [ ]:
training_args.per_device_train_batch_size = 16
training_args.per_device_eval_batch_size = 16
training_args.num_train_epochs = 10
training_args.metric_for_best_model = "f1_macro"
training_args.load_best_model_at_end = True
training_args.greater_is_better = True
training_args.evaluation_strategy = "epoch"
training_args.save_strategy = "epoch"
training_args.logging_strategy = "epoch"
training_args.save_total_limit = 2

### 3.8.1 1% train population

In [ ]:
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_1_pct.label, 
                      run_prefix="Final_model_sqrt_weighted",
                      num_train_epochs=15,
                      sqrt_weight=True
                      )

In [ ]:
results = test_model_over_populations_nn(
    model=trainer,
    dataset=datasets_test_encoded[model_name])

accuracy: 0.9747520288548241
precision: 0.7612587723264048
recall: 0.7704744242749217
f1_macro: 0.7509945043735637

- Accuracy only 0.0082pp difference from unweighted model
- F1-macro 0.5pp better than unweighted model.

### 3.8.2 10% sqrt train population

In [ ]:
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_10_pct_sqrt_weight.label, 
                      run_prefix="Final_model_sqrt_weighted",
                      num_train_epochs=15,
                      sqrt_weight=True
                      )

In [ ]:
results = test_model_over_populations_nn(
    model=trainer,
    dataset=datasets_test_encoded[model_name])

10% sqrt population
'accuracy': {'mean': 0.9751618985162718,
  'confidence_interval': {'low': 0.9725366833346996,
   'high': 0.9778690876301337}},
 'f1_macro': {'mean': 0.7822329552561796,
  'confidence_interval': {'low': 0.7554908144531797,
   'high': 0.8034555562942125}},

Accuracy and f1-macro are worse than unweighted model.

### 3.8.3 10% train population

In [ ]:
trainer = train_model(model_name=model_name, 
                      training_args=training_args, 
                      train_subset=DataSample.sample_10_pct.label, 
                      run_prefix="Final_model_sqrt_weighted",
                      num_train_epochs=15,
                      sqrt_weight=True
                      )

In [ ]:
results = test_model_over_populations_nn(
    model=trainer,
    dataset=datasets_test_encoded[model_name])

10% sqrt population
'accuracy': {'mean': 0.9751618985162718,
  'confidence_interval': {'low': 0.9725366833346996,
   'high': 0.9778690876301337}},
 'f1_macro': {'mean': 0.7822329552561796,
  'confidence_interval': {'low': 0.7554908144531797,
   'high': 0.8034555562942125}},

10% population
accuracy: 0.9757357160422986
precision: 0.7805558769944952
recall: 0.7877194041051261
f1_macro: 0.76625013027704


Accuracy and f1-macro are worse than unweighted model over 10% population.

Sqrt weighted model didn't perform better than unweighted model.

## 3.9. Random Oversampling


In [ ]:
import random
import math
from collections import Counter
from datasets import concatenate_datasets

def capped_oversample_dataset(
    dataset,
    label_col="label",
    max_factor=10,
    alpha=0.5,
    seed=SEED,
    verbose=True,
):
    rng = random.Random(seed)

    labels = [
    int(x.item()) if hasattr(x, "item") else int(x)
    for x in dataset[label_col]
]
    counts = Counter(labels)

    original_n = len(dataset)
    max_total_n = original_n * max_factor
    max_count = max(counts.values())

    target_counts = {}

    for cls, count in counts.items():
        target = count * ((max_count / count) ** alpha)

        # Important: ceil, not int floor
        target = math.ceil(target)

        # Never shrink a class
        target = max(count, target)

        target_counts[cls] = target

    total_target = sum(target_counts.values())

    # Respect global size cap
    if total_target > max_total_n:
        scale = max_total_n / total_target
        target_counts = {
            cls: max(counts[cls], math.floor(target_counts[cls] * scale))
            for cls in counts
        }

    if verbose:
        print("Original rows:", original_n)
        print("Target rows:", sum(target_counts.values()))
        print("Growth factor:", round(sum(target_counts.values()) / original_n, 3))
        print("Num classes:", len(counts))
        print("Smallest classes:", counts.most_common()[:-11:-1])
        print("Largest classes:", counts.most_common(10))

    chunks = [dataset]

    for cls, count in counts.items():
        extra_needed = target_counts[cls] - count

        if extra_needed <= 0:
            continue

        cls_indices = [
            i for i, y in enumerate(labels)
            if y == cls
        ]

        sampled_indices = [
            rng.choice(cls_indices)
            for _ in range(extra_needed)
        ]

        chunks.append(dataset.select(sampled_indices))

    return concatenate_datasets(chunks).shuffle(seed=seed)

In [ ]:
datasets_encoded[model_name]["sample_1_pct_oversampled"] = capped_oversample_dataset(datasets_encoded[model_name][DataSample.sample_1_pct.label], alpha = 0.7)

In [ ]:
datasets_encoded[model_name]["sample_1_pct_oversampled"] 

In [ ]:
datasets_encoded[model_name][DataSample.sample_1_pct.label]


### 3.9.1 1% train population oversampled

In [ ]:
training_args.per_device_train_batch_size = 16
trainer = train_model(
    model_name=model_name,
    training_args=training_args,
    train_subset="sample_1_pct_oversampled",
    run_prefix="Final_model",
    num_train_epochs=10,
)

0.078900	0.088032	0.974834	0.769832	0.774021	0.756782

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

1%
- accuracy: 0.9747520288548241
- precision: 0.7612587723264048
- recall: 0.7704744242749217
- f1_macro: 0.7509945043735637

Worse accuracy but better f1-macro score than 1% population

1% sqrt weighted population
- accuracy: 0.9745880809902451
- precision: 0.7946493848402242
- recall: 0.7987942120566772
- f1_macro: 0.779200270542408

Worse accuracy and f1-macro score than 1% sqrt weighted population

10% population
- 'accuracy': 0.9757357160422986,
- 'precision': 0.7805558769944952,
- 'recall': 0.7877194041051261,
- 'f1_macro': 0.76625013027704

Worse accuracy but f1-macro score than 10% population. 

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_test_encoded, test_name="holdout_10k")

In [ ]:
datasets_encoded[model_name]

### 3.9.2 1% sqrt weighted oversampled

In [ ]:
training_args.per_device_train_batch_size = 16
trainer = train_model(
    model_name=model_name,
    training_args=training_args,
    train_subset="sample_1_pct_sqrt_weight_oversampled",
    run_prefix="Final_model_weighted_oversampled",
    num_train_epochs=10,
)

1% sqrt weighted
'eval_accuracy': 0.9738503155996393, 'eval_precision': 0.7923935251750868, 'eval_recall': 0.7971910558477024, 'eval_f1_macro': 0.7758690249234124,

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_encoded, test_name="test_5_pct")

1% oversampled
{'accuracy': 0.9685220100008197,
 'precision': 0.770952953604877,
 'recall': 0.7889809680277952,
 'f1_macro': 0.7618247789851531}

1% sqrt population
accuracy: 0.9745880809902451
precision: 0.7946493848402242
recall: 0.7987942120566772
f1_macro: 0.779200270542408

Worse accuracy and but slightly better f1-macro than just oversampling 1% plain population.  
Worse accuracy and f1-macro score than 1% sqrt population.   

So oversampling made things worse

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_test_encoded, test_name="holdout_10k")

### 3.9.3 10% sqrt weighted oversampled

In [ ]:
datasets_encoded[model_name]["sample_10_pct_sqrt_weight_oversampled"] = capped_oversample_dataset(datasets_encoded[model_name][DataSample.sample_10_pct_sqrt_weight.label], alpha = 0.9)

In [ ]:
training_args.per_device_train_batch_size = 16
trainer = train_model(
    model_name=model_name,
    training_args=training_args,
    train_subset="sample_10_pct_sqrt_weight_oversampled",
    run_prefix="Final_model_weighted_oversampled",
    num_train_epochs=10,
)

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_test_encoded, test_name="test_5_pct")

Much worse f1-macro and accuracy than without oversampling.

In [ ]:
test_predictions(trainer=trainer, model_name=model_name, datasets_encoded=datasets_test_encoded, test_name="holdout_10k")

Overall normal oversampling gave worse results

# 4. Explainability

In [ ]:
pipe = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=AutoTokenizer.from_pretrained(model_name),
    device=trainer.args.device,
)

explainer = shap.Explainer(pipe)
shap_values = explainer(["cost of goods sold turnover"])
top_3 = (
    dataset_pl.filter(
        pl.col("canonical_label").is_in(["TurnoverRevenue", "CostSales", "RawMaterialsConsumablesUsed"])
    )
    .select("canonical_label", "label")
    .unique()["label"]
    .to_numpy()
)
for class_idx in top_3:
    class_name = trainer.model.config.id2label[class_idx]
    print(f"Class name {class_name}")

    shap.plots.text(shap_values[:, :, class_idx])

In [ ]:
dataset_pl.filter(pl.col("label").is_in(top_3)).select("canonical_label", "label").unique()

In [ ]:
shap.plots.text(shap_values)

When you click on a label, then the contributions of the input words show up at the bottom. 
Double click to show multiple labels
This is good in that you can see the contributions to CostOfSales, so turnover has a negative contribution. But with the Turnover category it's the opposite. 

In [ ]:
tokens = shap_values.data[0]
values = shap_values.values[0]

predicted_class_idx = values.sum(axis=0).argmax()

print(f"Predicted class: {pipe.model.config.id2label[predicted_class_idx]}")

for token, value in zip(tokens, values):
    contribution_to_prediction = value[predicted_class_idx]
    max_class_idx = value.argmax()
    max_class = pipe.model.config.id2label[max_class_idx]
    print(f"{token:20s}: towards prediction={contribution_to_prediction}: strongest pull towards {max_class} - {value.max():.4f}")

This is showing what each word pulls towards, with start and stop characters as well

In [ ]:
tokens = shap_values.data[0]
values = shap_values.values[0][:,predicted_class_idx]
print(values.shape)
sns.barplot(x=tokens, y=values)

This shows how much each word contributes to the final chosen label. This looks good since you'd expect cost to be related to CostOfSales, but Turnover isn't normally linked to that concept and here it has a negative contribution which is what you'd expect. 

While transformer models aren't as transparent to what they do, the SHAP package does provide sufficient levels of explainability. 

# 5. Loss over epochs

In [ ]:
logs = trainer.state.log_history
logs_pl = pl.DataFrame(logs)
ax = sns.lineplot(logs_pl, x="epoch", y="loss", label="train_loss")
sns.lineplot(logs_pl, x="epoch", y="eval_loss", ax=ax, label="eval_loss")
plt.title("Loss over epochs")

# 6. Investigate good and bad classification 

In [ ]:
def class_confusion_matrix(y_true, y_pred, target_class):

    y_true_bin = (y_true == target_class)
    y_pred_bin = (y_pred == target_class)

    cm = confusion_matrix(y_true_bin, y_pred_bin, labels=[True, False])

    return cm

In [ ]:
model_name = trainer.model.name_or_path
prediction_details = trainer.predict(datasets_encoded[model_name]["test_5_pct"])

In [ ]:
train_pl = dataset_pl.filter(pl.col("test_5_pct"))
y_pred = prediction_details.predictions.argmax(axis=1)
y_true = train_pl["label"].to_numpy()

In [ ]:
train_pl.with_columns(pl.Series("pred_label", y_pred), pl.Series("prediction_label", le.inverse_transform(y_pred))).filter(pl.col("pred_label") != pl.col("label"))

In [ ]:
le = LabelEncoder()
le.classes_ = dataset_pl.select("label", "canonical_label").unique().sort("label")["canonical_label"]

In [ ]:
le.inverse_transform(y_pred)

In [ ]:
class_conf_matrix = class_confusion_matrix(y_true, y_pred, 127)

In [ ]:
for i in range(len(le.classes_)):
  class_matrix = class_confusion_matrix(y_true, y_pred, i)
  if(class_matrix[0][0] > 10 and class_matrix[0][1] > 1):
    print(i, le.inverse_transform([i])[0], class_matrix)

In [ ]:
def plot_confusion_matrix_heatmap(y_true, y_pred, target_class, figsize=(12, 10), normalize=False):
    cm = class_confusion_matrix(y_true, y_pred, target_class)
    print(cm)

    fig, ax = plt.subplots(figsize=figsize)

    # Create heatmap with better spacing
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Positive', 'Negative'],
                yticklabels=['Positive', 'Negative'],
                ax=ax, cbar_kws={'label': 'Count' if not normalize else 'Proportion'},
                square=True,
                annot_kws={'size': 12})

    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')
    ax.set_title(le.inverse_transform([target_class])[0] + " Confusion Matrix")

    plt.xticks(rotation=0)
    plt.yticks(rotation=0)
    plt.tight_layout()

    return fig, ax



In [ ]:
plot_confusion_matrix_heatmap(y_true, y_pred, 73)
plot_confusion_matrix_heatmap(y_true, y_pred, 149)